# 20.19. MP-02 — маршрут маска → поверхность → FEM-геометрия

**Статус:** технический pilot. Проверена поверхность полного MONAI Physio highres union `140–144` и одноимённого проектного raw highres union, оба без морфологической обработки и с `stride=1`. У кандидата четыре поверхностные компоненты, поэтому тетраэдризация не запускалась. Ранее проектный `whole_heart` отдельно блокировался в Gmsh; MONAI Physio VTK-выход не проверен.

**Постановка.** Сначала зафиксировать одну бинарную маску, affine RAS, мм, фазу/дыхание и её контрольную сумму. Исходный маршрут проекта — marching cubes → STL (`cardiac4d.export_mesh`). Сравнивать поверхности MONAI Physio только при той же исходной маске; иначе различие алгоритма сегментации смешается с различием геометрического шага. STL не хранит единицы, поэтому координатный контракт задаётся отдельно.

**Критерии.** Объём вокселей и замкнутой поверхности, число связных компонент, winding, Эйлерова характеристика, качество и знак тетраэдров. Для полноценного FEM-сравнения нужны одинаковые тканевые границы, конечные электроды, материалы, контактные условия, объёмная сетка и расчёты Z/ΔZ; один STL не достаточен.

**Измерения на фазе 00 Nix (28.09.2026).** Полный MONAI highres union: 569,387 мл по вокселям, 129894 вершин и 259788 треугольников; поверхность замкнута, winding согласован, площадь 49638,55 мм², объём 569,305 мл, 4 поверхностные компоненты, Эйлер 0. Проектный raw highres union тех же пяти классов: 571,987 мл, 130524 вершин и 261052 треугольника; поверхность замкнута, площадь 49175,80 мм², объём 571,912 мл, 1 поверхностная компонента, Эйлер −2. Обе воксельные маски 6-связны. Различие площадей 462,75 мм² связано с разными входными масками при одном marching cubes и не является метрикой электрического ответа.

**Межклассовая граница.** Проектный очищенный `whole_heart` — другой объект (307036 треугольников, 67206,76 мм², 3 поверхностные компоненты); его прежний Gmsh blocker не переносится автоматически на highres union. Для полного MONAI union необходим разбор дополнительных поверхностных компонент до Gmsh. FEM `Z`/`ΔZ` не рассчитывались. Единица NIfTI `unknown`, мм использованы только по явному допущению из происхождения CT.


## Основной MP-02 на Нике: вход в одинаковую FEM-постановку

[Машинный preflight](mp02_nix_fem_preflight.json) из [скрипта](mp02_nix_fem_preflight.py) сопоставил результаты полноразрешённой регистрации Ника с уже существующей индивидуальной FEM-постановкой. Хэши автоматических масок фаз 00, 01 и 07 совпадают с использованными в FEM-фазах. Материализация и план используют одну подготовленную сетку (`791 141` узел, `4 448 426` тетраэдров), одинаковый фон, CEM-контакты диаметром 5 мм, 50 кГц; у плана 72 решения на 12 фаз. Общий DICOM patient frame статической и динамической серий установлен, но независимой оценки ошибки карты КТ→FEM нет.

[Полноразрешённый отчёт](mp03_nix_fullres_report.md) показывает технически положительные поля 00→01 и сглаженное 00→07. Ни одно не допущено в FEM: отсутствует принятая ручная разметка именно динамических фаз и независимая оценка ошибки карты КТ→FEM. [Верхний контакт](mp03_nix_fov_contact_report.md) относится к наружной маске `whole_heart`; кровь и миокард до границы не доходят, роль этого участка в FEM требует отдельной оценки. Прежние разделы этого ноутбука — пилот сравнения поверхностей; **нового парного расчёта `Z`/`ΔZ` не было**. При допуске допустимой геометрии менять разрешено только способ задания сердечной области, оставляя ту же сетку, материалы, проводимости, контакты, частоту и решатель.


## Решение после полного технического экрана 11 фаз Ника

[Регистрация всех целевых фаз](mp03_nix_full_cycle_report.md) и перенос исходной поверхности выполнены как внутренний расчёт: сглаженные поля не имеют неположительного якобиана на проверенной сетке, а расстояние до отдельных автоматических целевых масок в среднем уменьшается. Это не снимает анатомический барьер. Принятой независимой ручной фазовой разметки нет, наружная автоматическая маска `whole_heart` касается границы FOV во всех фазах, а ошибка карты КТ→FEM не ограничена независимо. Кровь и миокард по отдельным auto-маскам верхней грани не достигают.

Поэтому **ни одна новая перенесённая геометрия не поступила в существующую одинаковую FEM-постановку**. [FEM-preflight](mp02_nix_fem_preflight.json) описывает ранее построенную модель на отдельных автоматических масках и её параметры, но не содержит парного расчёта `Z`/`ΔZ` с проверенным полем. Вывод о замене геометрии или сокращении входов пока невозможен.

In [ ]:
from datetime import datetime, timezone
import os, subprocess, sys
from pathlib import Path
mask = os.environ.get('MP02_MASK')
if not mask:
    print('Set MP02_MASK to a local binary NIfTI mask')
else:
    output = Path('runs') / f"mp01_02_geometry_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}.json"
    command = [sys.executable, 'mp01_02_pilot.py', 'geometry', '--mask', mask, '--stride', os.environ.get('MP02_STRIDE', '1'), '--output', str(output)]
    if os.environ.get('MP02_VERIFIED_MM') == '1': command.append('--assume-mm')
    if os.environ.get('MP02_REFERENCE_STL'): command.extend(['--reference-surface', os.environ['MP02_REFERENCE_STL']])
    if os.environ.get('MP02_TETRA') == '1': command.append('--tetra')
    subprocess.run(command, check=True)


In [ ]:
# Reproduce the paired highres geometry QC using local paths only.
# Set MP02_CT, MP02_BASELINE_HIGHRES_DIR and MP02_FULL_CANDIDATE_DIR locally.
import hashlib, json
from datetime import datetime, timezone
import nibabel as nib
import numpy as np
from mp01_02_compare_heart import LABELS, binary, sha256
keys = ('MP02_CT', 'MP02_BASELINE_HIGHRES_DIR', 'MP02_FULL_CANDIDATE_DIR')
missing = [key for key in keys if not os.environ.get(key)]
if missing:
    print('Local inputs not configured:', ', '.join(missing))
else:
    ct_path = Path(os.environ['MP02_CT'])
    candidate_dir = Path(os.environ['MP02_FULL_CANDIDATE_DIR'])
    manifest = json.loads((candidate_dir / 'mp01_02_manifest.json').read_text(encoding='utf-8'))
    assert manifest['full_monai_physio_workflow'] is True
    assert manifest['executed_tasks'] == ['total', 'heartchambers_highres', 'tissue_4_types', 'lung_vessels', 'body']
    assert manifest['ct_sha256'] == sha256(ct_path)
    ct = nib.load(str(ct_path))
    baseline_union = np.zeros(ct.shape, dtype=bool)
    for filename in LABELS.values():
        baseline_union |= binary(Path(os.environ['MP02_BASELINE_HIGHRES_DIR']) / filename, ct)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    root = Path('runs').resolve()
    baseline_path = root / f'mp01_02_project_highres_union_{stamp}.nii.gz'
    assert not baseline_path.exists()
    image = nib.Nifti1Image(baseline_union.astype(np.uint8), ct.affine, ct.header)
    image.set_data_dtype(np.uint8)
    nib.save(image, str(baseline_path))
    for route, mask_path in [('project', baseline_path), ('full_monai', candidate_dir / 'mp01_02_highres_heart_union.nii.gz')]:
        output = root / f'mp01_02_{route}_highres_geometry_{stamp}.json'
        subprocess.run([sys.executable, 'mp01_02_pilot.py', 'geometry', '--mask', str(mask_path), '--stride', '1', '--assume-mm', '--output', str(output)], check=True)
        result = json.loads(output.read_text(encoding='utf-8'))
        print(route, result['original_mask'], result['surface'])
    # Run Gmsh separately only after verifying one valid connected boundary.


**Границы вывода.** `stride>1`, closing и заполнение полостей меняют анатомическую геометрию и могут скрыть локальные ошибки. Это отдельные явно подписанные сценарии. Внутренний QC и синтетический опыт не подтверждают анатомическую точность. До сравнения FEM-отклика при одинаковых параметрах результат не передавать в модель ТТРКГ как принятую геометрию. Локальные маски, STL, MSH и JSON пишутся только в исключённый `runs/`.
